In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error

# Trying out logistic regression
## lets see the rmse values

In [2]:
def read_dataframe(filename):
    df=pd.read_parquet(filename)
    df.tpep_pickup_datetime=pd.to_datetime(df.tpep_pickup_datetime)
    df.tpep_dropoff_datetime=pd.to_datetime(df.tpep_dropoff_datetime)
    df['duration']=(df.tpep_dropoff_datetime)-(df.tpep_pickup_datetime)
    df.duration=df.duration.apply(lambda td: td.total_seconds()/60)
    df=df[(df.duration >=1)&(df.duration <=60)]
    categorical=['PULocationID','DOLocationID']
    ##numerical=['trip_distance']
    df[categorical]=df[categorical].astype(str)
    
    return df

In [3]:
df_train= read_dataframe(r'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet')
df_val=read_dataframe(r'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet')

In [4]:
len(df_train),len(df_val)

(3009173, 2855951)

In [5]:
df_train.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,duration
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00,8.433333
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00,6.316667
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00,12.750000
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25,9.616667
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00,10.833333


In [6]:
df_train.duration.describe() ##std for duration

count    3.009173e+06
mean     1.420486e+01
std      9.939386e+00
min      1.000000e+00
25%      7.216667e+00
50%      1.155000e+01
75%      1.818333e+01
max      6.000000e+01
Name: duration, dtype: float64

In [4]:
def model_run(df):
    train_dict=df[categorical+numerical].to_dict(orient='records')
    dv=DictVectorizer()

    X_train=dv.fit_transform(train_dict)
    y_train=df['duration'].values

    lr=LogisticRegression()
    lr.fit(X_train,y_train)

    y_pred=lr.predict(X_train)

    return mean_squared_error(y_train,y_pred,squared=False)
    

In [5]:
categorical=['PULocationID','DOLocationID']
numerical=['trip_distance']

In [6]:
rmse_train=model_run(df_train)
rmse_val=model_run(df_val)
print('rmse_train')
print(rmse_train)
print('rmse_val')
print(rmse_val)

ValueError: Unknown label type: 'continuous'